# BME 574 — Project 5 preparation
## Scoring the unusual, and checking whether it was worth it

Read and run. Nothing here is handed in; it exists so that the starter notebook is not
the first place you meet these ideas. About twenty-five minutes.

By the end you should be able to say, in your own words:

1. why training on normal cases only is a different problem from classification;
2. why reconstruction error and *probability* are not the same thing;
3. what the $\sigma^2$ in a Gaussian decoder does to your anomaly score;
4. why a lower bound is fine for ranking and dangerous for reporting;
5. and why the first thing to do with any anomaly detector is race it against a
   one-line baseline.

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from sklearn.decomposition import PCA

torch.set_num_threads(4)
rng = np.random.default_rng(0)
print('ready')

---
## 1 · The setup: you are never shown an anomaly

In Projects 3 and 4 you had labels for every class and the model learned the boundary
between them. Here the training set contains **one** class — the normal one — and the
model's only job is to learn what normal looks like well enough that abnormal scores
differently.

That changes what can go wrong. A classifier fails by drawing the boundary in the wrong
place. A density model fails by being *too good*: if it can reconstruct anything you show
it, including things it has never seen, it will not flag them.

In [ ]:
# A two-dimensional example we can actually look at.
n = 1200
t = rng.uniform(0, 2 * np.pi, n)
r = 2.0 + rng.normal(0, 0.12, n)
normal = np.stack([r * np.cos(t), r * np.sin(t)], 1)        # a ring

a_out = rng.normal(0, 0.35, (60, 2)) + np.array([3.6, 0.0])  # far outside
a_in = rng.normal(0, 0.18, (60, 2))                          # in the HOLE

plt.figure(figsize=(4.6, 4.6))
plt.scatter(*normal.T, s=6, color='#8899AA', label='normal (training)')
plt.scatter(*a_out.T, s=16, color='#9C2B2B', label='anomaly: outside')
plt.scatter(*a_in.T, s=16, color='#B9733A', label='anomaly: in the hole')
plt.legend(fontsize=8); plt.gca().set_aspect('equal'); plt.title('Figure 1 — the ring')
plt.tight_layout(); plt.show()
print('Both red and amber points are abnormal. Only one of them is far from the data.')

---
## 2 · Reconstruction error is not probability

A plain autoencoder scores by how well it can rebuild the input. On the ring, the hole in
the middle is exactly the place a 1-D bottleneck will happily map onto — so points there
can be reconstructed *well* while being wildly improbable.

Watch a linear model with one component make this mistake.

In [ ]:
p = PCA(n_components=1).fit(normal)


def recon_err(A):
    return ((A - p.inverse_transform(p.transform(A))) ** 2).sum(1)


def maha(A):
    m = normal.mean(0)
    P = np.linalg.inv(np.cov(normal.T) + 1e-6 * np.eye(2))
    D = A - m
    return np.einsum('ij,jk,ik->i', D, P, D)


for name, fn in (('PCA reconstruction error', recon_err), ('Mahalanobis', maha)):
    print('%-26s  normal %6.3f | outside %6.3f | in the hole %6.3f'
          % (name, fn(normal).mean(), fn(a_out).mean(), fn(a_in).mean()))
print()
print('Both scores call the outside points unusual.')
print('Neither of them minds the ones sitting in the hole - and a doughnut hole is')
print('precisely where a patient who is abnormal in a *combination* of measurements')
print('would land.')

---
## 3 · What the VAE adds, and what it charges for it

A VAE scores with the **negative ELBO**, which is reconstruction error *plus* a term that
asks whether the code it assigned you looks like the codes it assigned the training data:

$$-\mathcal{L}(x) = \underbrace{-\mathbb{E}_q\!\left[\log p_\theta(x\mid z)\right]}_{\text{can I rebuild it}}
\;+\; \underbrace{D_{KL}\!\left(q_\phi(z\mid x)\,\|\,p(z)\right)}_{\text{is its code normal}}$$

The second term is what a plain autoencoder does not have. Let us see whether it helps on
the ring.

In [ ]:
class TinyVAE(nn.Module):
    def __init__(self, d=2, z=1, h=64, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.enc = nn.Sequential(nn.Linear(d, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU())
        self.mu, self.lv = nn.Linear(h, z), nn.Linear(h, z)
        self.dec = nn.Sequential(nn.Linear(z, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(),
                                 nn.Linear(h, d))

    def forward(self, x):
        hh = self.enc(x); mu, lv = self.mu(hh), self.lv(hh)
        z = mu + torch.exp(0.5 * lv) * torch.randn_like(mu)
        return self.dec(z), mu, lv


def parts(x, xh, mu, lv, s2):
    d = x.shape[1]
    rec = -0.5 * (((x - xh) ** 2).sum(1) / s2 + d * math.log(2 * math.pi * s2))
    kl = -0.5 * (1 + lv - mu.pow(2) - lv.exp()).sum(1)
    return rec, kl


def train(X, s2, epochs=400, seed=0):
    m = TinyVAE(seed=seed); opt = torch.optim.Adam(m.parameters(), lr=3e-3)
    Xt = torch.tensor(X, dtype=torch.float32)
    for _ in range(epochs):
        xh, mu, lv = m(Xt)
        rec, kl = parts(Xt, xh, mu, lv, s2)
        loss = -(rec - kl).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    return m.eval()


@torch.no_grad()
def score(m, X, s2, k=16):
    x = torch.tensor(X, dtype=torch.float32); tot = torch.zeros(len(x))
    for _ in range(k):
        xh, mu, lv = m(x); rec, kl = parts(x, xh, mu, lv, s2)
        tot += -(rec - kl)
    return (tot / k).numpy()


vae = train(normal, 0.5)
test = np.concatenate([normal[:300], a_out, a_in])
lab = np.concatenate([np.zeros(300), np.ones(len(a_out) + len(a_in))])
lab_hole = np.concatenate([np.zeros(300), np.ones(len(a_in))])
hole_only = np.concatenate([normal[:300], a_in])

print('AUC, all anomalies     VAE %.3f   PCA %.3f   Mahalanobis %.3f'
      % (roc_auc_score(lab, score(vae, test, 0.5)),
         roc_auc_score(lab, recon_err(test)), roc_auc_score(lab, maha(test))))
print('AUC, hole only         VAE %.3f   PCA %.3f   Mahalanobis %.3f'
      % (roc_auc_score(lab_hole, score(vae, hole_only, 0.5)),
         roc_auc_score(lab_hole, recon_err(hole_only)),
         roc_auc_score(lab_hole, maha(hole_only))))

Read those two lines carefully. Pooled over both kinds of anomaly everything looks
respectable. Split out the hard case and the picture changes completely.

**This is the single most important habit in this project: report your score broken down
by the kind of anomaly, not pooled.** A pooled AUC is an average over whatever mix
happened to be in your test set, and in a clinical setting the rare, subtle, hard class is
usually the one that matters.

---
## 4 · The $\sigma^2$ you did not know you were choosing

The Gaussian reconstruction term is

$$\log p_\theta(x\mid z) = -\frac{1}{2}\sum_j \left[\frac{(x_j-\hat{x}_j)^2}{\sigma^2}
+ \log 2\pi\sigma^2\right]$$

Write "sum of squared errors" as your reconstruction loss and you have set
$\sigma^2 = 1/2$. That is a claim: *the measurement noise on every feature has variance
one half*. On standardised data whose residuals sit at 0.05, it is wrong by a factor of
ten, and the factor lands directly on the balance between the two loss terms.

Here is the same model trained at three different $\sigma^2$, scored on the same points.

In [ ]:
# A SUBTLE anomaly: still on the ring, but at slightly the wrong radius. This is the
# realistic case - the obvious outliers were never the hard part.
r_sub = 2.0 + rng.normal(0, 0.12, 200) + 0.42
t_sub = rng.uniform(0, 2 * np.pi, 200)
subtle = np.stack([r_sub * np.cos(t_sub), r_sub * np.sin(t_sub)], 1)
X_sub = np.concatenate([normal[:400], subtle])
lab_sub = np.concatenate([np.zeros(400), np.ones(200)])

s2s = [0.5, 0.2, 0.05, 0.01]
aucs = []
for s2 in s2s:
    m = train(normal, s2)
    aucs.append(roc_auc_score(lab_sub, score(m, X_sub, s2)))
    print('sigma^2 = %.2f   AUC on the subtle anomalies %.3f' % (s2, aucs[-1]))
print()
print('Same data, same architecture, same number of epochs. The only thing that')
print('changed is a constant you may not have realised you were setting.')

plt.figure(figsize=(5.4, 3.0))
plt.semilogx(s2s, aucs, 'o-', color='#1F3864', lw=2.2, ms=6)
plt.axvline(0.5, color='#9C2B2B', ls='--', lw=1.5)
plt.text(0.52, min(aucs) + 0.01, 'what plain SSE assumes', color='#9C2B2B', fontsize=8.5)
plt.xlabel('$\\sigma^2$ used in the loss'); plt.ylabel('AUC')
plt.title('Figure 2 — the same model, four observation variances')
plt.grid(alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
# What the residuals actually say.
m = train(normal, 0.5)
with torch.no_grad():
    x = torch.tensor(normal, dtype=torch.float32)
    hh = m.enc(x); xh = m.dec(m.mu(hh))
    s2_hat = float(((x - xh) ** 2).mean())
print('fitted sigma^2 from the residuals : %.4f' % s2_hat)
print('what a plain SSE loss assumed     : 0.5')
print('so the KL term was over-weighted by about %.0fx' % (0.5 / s2_hat))

---
## 5 · A bound is not a number

The ELBO is a *lower* bound on $\log p_\theta(x)$, so $-$ELBO is an *upper* bound on the
surprise. The gap is the KL divergence between the encoder's guess and the true posterior,
and it is different for every example.

For **ranking** that is usually survivable, because the gap is small and roughly similar
across examples. For **reporting an absolute number** it is not: "this patient has
log-likelihood $-84$" is false; the honest statement is "at most $-84$, and I have not
measured by how much."

You can see the gap shrink by spending more compute on it.

In [ ]:
@torch.no_grad()
def iwae(m, X, s2, L):
    'Importance-weighted estimate: L=1 is the ELBO, larger L climbs toward log p(x).'
    x = torch.tensor(X, dtype=torch.float32)
    hh = m.enc(x); mu, lv = m.mu(hh), m.lv(hh)
    lw = []
    for _ in range(L):
        eps = torch.randn_like(mu); z = mu + torch.exp(0.5 * lv) * eps
        xh = m.dec(z)
        d = x.shape[1]
        lpxz = -0.5 * (((x - xh) ** 2).sum(1) / s2 + d * math.log(2 * math.pi * s2))
        lpz = -0.5 * (z ** 2 + math.log(2 * math.pi)).sum(1)
        lqz = -0.5 * ((z - mu) ** 2 / lv.exp() + lv + math.log(2 * math.pi)).sum(1)
        lw.append(lpxz + lpz - lqz)
    lw = torch.stack(lw)
    return float((torch.logsumexp(lw, 0) - math.log(L)).mean())


m = train(normal, 0.1)
for L in (1, 5, 50, 500):
    print('L = %4d   estimated log p(x) = %8.4f' % (L, iwae(m, normal[:400], 0.1, L)))
print()
print('L = 1 IS the ELBO. Every later row is the same model, measured better.')
print('The climb is the bound tightening - the model did not improve.')

---
## 6 · Race it against something trivial, every time

Finally, the habit the whole project is built around. Three baselines, each a few lines,
each fitted on the training normals only:

| baseline | what it assumes |
|---|---|
| Mahalanobis | the normal data is one Gaussian blob |
| PCA reconstruction | the normal data lies near a linear subspace |
| one-class SVM | the normal data fills some compact region |

A VAE should beat these when the normal data is genuinely curved, clustered, or
multi-modal. When it does not, that is not a failed experiment — it is a finding about
your data, and it belongs in the write-up.

In [ ]:
# A dataset where the trivial baseline wins, so that you have seen one.
blob = rng.normal(0, 1, (1200, 8)) @ rng.normal(0, 1, (8, 8))
far = rng.normal(0, 1, (120, 8)) @ rng.normal(0, 1, (8, 8)) * 2.6
Xt_ = np.concatenate([blob[:400], far]); lab_ = np.concatenate([np.zeros(400), np.ones(120)])

mb, Pb = blob.mean(0), np.linalg.inv(np.cov(blob.T) + 1e-6 * np.eye(8))
m_auc = roc_auc_score(lab_, np.einsum('ij,jk,ik->i', Xt_ - mb, Pb, Xt_ - mb))


class V8(TinyVAE):
    def __init__(self, seed=0):
        super().__init__(d=8, z=3, seed=seed)


def train8(X, s2, epochs=400, seed=0):
    m = V8(seed); opt = torch.optim.Adam(m.parameters(), lr=3e-3)
    Xt2 = torch.tensor(X, dtype=torch.float32)
    for _ in range(epochs):
        xh, mu, lv = m(Xt2); rec, kl = parts(Xt2, xh, mu, lv, s2)
        loss = -(rec - kl).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    return m.eval()


v8 = train8(blob, 0.1)
v_auc = roc_auc_score(lab_, score(v8, Xt_, 0.1))
print('one Gaussian, three lines of numpy : AUC %.3f' % m_auc)
print('a variational autoencoder          : AUC %.3f' % v_auc)
print()
print('The data really is one Gaussian blob, so the Gaussian wins. Nothing has gone')
print('wrong - the VAE simply had nothing extra to find, and cost you an afternoon.')

---
## You are ready for the starter when you can answer these

1. Why does the training split in this project contain no anomalies at all?
2. Give an example of a point that is easy to reconstruct and still very improbable.
3. You write your reconstruction loss as `((x - xhat)**2).sum(1)`. What have you just
   assumed about the measurement noise?
4. Your model reports $-$ELBO $= 84.2$ for a patient. What is the honest one-sentence
   version of that statement?
5. Your VAE gets AUC 0.91 and a one-class SVM gets 0.93, and your three seeds span
   $\pm 0.01$. What do you write in the memo?